In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")
print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
data_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
missing_values = df.isnull().sum()
print(f"Missing rows data: {missing_values[missing_values > 0]}")

null_columns = [val for val in missing_values.keys()] # columns that contains null values
clean_df = df[null_columns].fillna(df[null_columns].mode())
clean_df = df[null_columns].fillna(df[null_columns].mean())


print(f"\nMissing rows data: {clean_df.isnull().sum().sum()}\n\n")
clean_df.head()

In [ ]:
# Task 2: Write your code here:
duplicates = df.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
  print("Dropping Duplicates...")
  df.drop_duplicates(inplace=True)
  print("Duplicates Dropped.")
else:
  print("No Duplicate Samples Found.")

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Encode features and target using LabelEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns
if len(categorical_cols) > 0:
  for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    clean_df[col] = le.fit_transform(clean_df[col])
else:
  print("No categorical columns found.\n\n")

clean_df.head()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_cols = clean_df.drop("Target", axis=1).columns # Another way to extract features columns than i've did in Q1.
clean_df[features_cols] = scaler.fit_transform(clean_df[features_cols])
clean_df.head()

In [ ]:
# Task 5: Write your code here:

# Show full dataset class distribution
full_ratio = (df['Target'].value_counts(normalize=True) * 100).sort_index()
print("Full Dataset Class Distribution")
print("  Target classes percentages:", {k: f"{v:.2f}%" for k, v in full_ratio.items()})
print("-" * 40, '\n')

df["Target"].hist() # it's imbalanced data.

In [ ]:
# Task 1: Write your code here:
X = clean_df.drop("Target", axis=1)
y = clean_df["Target"]

print(X.shape, y.shape)

In [ ]:
pip install catboost # it wasn't installed :!


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold # is the correct one :)
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, accuracy_score



accuracy_scores = []
f1_scores = []
model = CatBoostClassifier()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    accuracy_scores.append(accuracy_score(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred))

print(f"Avg Accuracy loss: {(sum(accuracy_scores) / 5)}") # i divide the total losses on the number of Folds = 5
print(f"Avg F1 loss: {(sum(f1_scores) / 5)}")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

features_for_importance = [col for col in features_cols if col != 'is_legendary']
feature_importance = pd.DataFrame({
    'feature': features_for_importance,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
clean_df["P_2"].hist() # This is the one!

In [ ]:
# Task Bonus: Write your code here: